In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

def optimize_and_pickle(csv_path, pkl_path):
    print(f"--- Starting Optimization for {csv_path} ---")
    
    # 1. Read the CSV
    # usecols: Only load what you need to save RAM immediately
    # low_memory=False: Prevents type-guessing errors on large files
    df = pd.read_csv(csv_path, low_memory=False)
    
    initial_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Initial Memory Usage: {initial_mem:.2f} MB")

    # 2. Downcast Logic
    for col in df.columns:
        col_type = df[col].dtype

        if col_type == 'object':
            # Convert strings with low cardinality to 'category'
            # (e.g., synthetic_zone or gender)
            num_unique = df[col].nunique()
            if num_unique / len(df) < 0.5:
                df[col] = df[col].astype('category')
        
        elif 'int' in str(col_type):
            # Downcast to smallest possible integer (int8, int16, etc.)
            df[col] = pd.to_numeric(df[col], downcast='integer')
            
        elif 'float' in str(col_type):
            # Downcast to float32
            df[col] = pd.to_numeric(df[col], downcast='float')

    final_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Optimized Memory Usage: {final_mem:.2f} MB")
    print(f"Reduction: {((initial_mem - final_mem) / initial_mem) * 100:.1f}%")

    # 3. Save as Pickle
    # Using protocol 5 (fastest for Python 3.8+)
    os.makedirs(os.path.dirname(pkl_path), exist_ok=True)
    df.to_pickle(pkl_path, protocol=5)
    print(f"Successfully saved to: {pkl_path}")

# --- EXECUTION ---
optimize_and_pickle('../data/sipher/sipher.csv', '../data/sipher/pickles/sipher_optimized.pkl')

--- Starting Optimization for sipher.csv ---
Initial Memory Usage: 3326.76 MB
Optimized Memory Usage: 3125.14 MB
Reduction: 6.1%
Successfully saved to: sipher_optimized.pkl
